# 05 Evaluation

This notebook evaluates the poetry graph pipeline from the saved annotation files and processed project outputs.

The evaluation folder only stores gold standard or annotation CSV files. Metric tables are calculated inside this notebook and are not saved as separate CSV files.


## Annotation Files

The notebook uses these annotation files.

- blind_gold_poem_sample.csv contains sampled poems with proxy gold symbols and emotion categories.
- relation_annotation_sample.csv contains sampled symbol-emotion relations with quality labels.
- retrieval_evaluation_sample.csv contains retrieved poems with relevance labels.

These files are kept because they are the evidence behind the evaluation. Summary metrics are calculated live from them.


In [1]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
EVALUATION_DIR = ROOT / 'outputs' / 'evaluation'

POEMS_PATH = PROCESSED_DIR / 'poems_clean.csv'
SYMBOLS_PATH = PROCESSED_DIR / 'extracted_symbols.csv'
EMOTIONS_PATH = PROCESSED_DIR / 'extracted_emotions.csv'
RELATIONS_PATH = PROCESSED_DIR / 'symbol_emotion_edges.csv'
GRAPH_NODES_PATH = PROCESSED_DIR / 'graph_nodes.csv'
GRAPH_EDGES_PATH = PROCESSED_DIR / 'graph_edges.csv'

pd.set_option('display.max_colwidth', 140)


## Helper Functions

These functions normalize labels and calculate precision, recall, and F1 from item sets.


In [2]:
def normalize_value(value):
    text = re.sub(r'\s+', ' ', str(value).lower()).strip()
    return re.sub(r'^[^a-z]+|[^a-z]+$', '', text)

def split_label_list(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {normalize_value(item) for item in str(value).split(',') if normalize_value(item)}

def set_metrics(gold_sets, predicted_sets):
    rows = []
    total_true_positive = 0
    total_false_positive = 0
    total_false_negative = 0
    for poem_id, gold in gold_sets.items():
        predicted = predicted_sets.get(poem_id, set())
        true_positive = len(gold & predicted)
        false_positive = len(predicted - gold)
        false_negative = len(gold - predicted)
        total_true_positive += true_positive
        total_false_positive += false_positive
        total_false_negative += false_negative
        rows.append({
            'poem_id': poem_id,
            'true_positive': true_positive,
            'false_positive': false_positive,
            'false_negative': false_negative,
            'gold_count': len(gold),
            'predicted_count': len(predicted),
            'matched_items': ', '.join(sorted(gold & predicted)),
            'missed_items': ', '.join(sorted(gold - predicted)),
            'extra_items': ', '.join(sorted(predicted - gold))
        })
    precision = total_true_positive / (total_true_positive + total_false_positive) if total_true_positive + total_false_positive else 0
    recall = total_true_positive / (total_true_positive + total_false_negative) if total_true_positive + total_false_negative else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    summary = pd.DataFrame([{
        'true_positive': total_true_positive,
        'false_positive': total_false_positive,
        'false_negative': total_false_negative,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }])
    return summary, pd.DataFrame(rows)


## Gold Standard Sample

This table is the retained gold standard file for symbol and emotion evaluation. The labels were created from poem text without using the extracted output columns.


In [3]:
gold_sample = pd.read_csv(EVALUATION_DIR / 'blind_gold_poem_sample.csv')
gold_sample[['poem_id', 'title', 'author', 'gold_symbols', 'gold_emotion_categories', 'annotation_method']].head(10)


,poem_id,title,author,gold_symbols,gold_emotion_categories,annotation_method
0,poem_11269,Minor Poet,Bill Sweeney,"mother, night, head, tide, beach, year, plague, brother","hope, joy, mortality",blind_objective_proxy_from_poem_text
1,poem_05452,Wild Poppies,Marion McCready,"throat, bull, night, neck, silk, breadth, pyrenees, opera","despair, envy, freedom, hope, love, melancholy, resilience",blind_objective_proxy_from_poem_text
2,poem_12480,I Heard an Angel,William Blake,"sun, curse, hay, haycock, devil, heath, pourd, grain","freedom, joy, melancholy, peace, tenderness, transcendence",blind_objective_proxy_from_poem_text
3,poem_06414,An Anthology of Rain,Phillis Levin,"drop, collection, limit, air, water, need, devoid, value","anxiety, freedom, gratitude, melancholy, mortality, peace",blind_objective_proxy_from_poem_text
4,poem_11839,Sonnet of the Seven Chinese,Franco Fortini,"image, wall, poet, room, print, photo, worker, lens","doubt, nostalgia",blind_objective_proxy_from_poem_text
5,poem_09632,"from Inscriptions, 16: ""The lamps are burning in the synagogue""",Charles Reznikoff,"morning, eli, answer, lamp, synagogue, house, study, alley","despair, doubt, faith, fear, hope, love, mortality, nostalgia, regret, resilience, transcendence, wonder",blind_objective_proxy_from_poem_text
6,poem_10031,In Memoriam Mae Noblitt,A. R. Ammons,"current, reality, atmosphere, season, space, heel, silk, air","anxiety, fear, freedom, grief, hope, longing, love, nostalgia",blind_objective_proxy_from_poem_text
7,poem_01828,Incident,Natasha Trethewey,"story, window, grass, cross, tree, room, lamp, gown","envy, fear, hope, melancholy, nostalgia, peace, transcendence",blind_objective_proxy_from_poem_text
8,poem_01791,Conduct,Samuel Greenberg,"peninsula, painter, grove, apostle, alm, meek, volcano, sulphur","desire, longing, peace, transcendence",blind_objective_proxy_from_poem_text
9,poem_11907,“Some motionless conﬂict in the sky...”,Donald Revell,"cloud, motionless, skya, universalsimply, saidshe, color, house, rochester","joy, transcendence",blind_objective_proxy_from_poem_text


## Symbol Extraction Evaluation

The system symbols are compared with the gold symbols from the sampled poems. The metric table is calculated here and not saved separately.


In [4]:
symbols_df = pd.read_csv(SYMBOLS_PATH)
gold_symbol_sets = dict(zip(gold_sample['poem_id'], gold_sample['gold_symbols'].map(split_label_list)))
predicted_symbol_sets = symbols_df[symbols_df['poem_id'].isin(gold_symbol_sets)].groupby('poem_id')['symbol'].apply(lambda values: {normalize_value(value) for value in values.dropna() if normalize_value(value)}).to_dict()
symbol_summary, symbol_details = set_metrics(gold_symbol_sets, predicted_symbol_sets)
display(symbol_summary)
symbol_details.head(10)


,true_positive,false_positive,false_negative,precision,recall,f1
0,121,172,119,0.412969,0.504167,0.454034


,poem_id,true_positive,false_positive,false_negative,gold_count,predicted_count,matched_items,missed_items,extra_items
0,poem_11269,2,8,6,8,10,"mother, plague","beach, brother, head, night, tide, year","aegean, eave, elegies, magazine, pacific, steamer, vigil, weakness"
1,poem_05452,3,8,5,8,11,"breadth, opera, silk","bull, neck, night, pyrenees, throat","anemone, butterfly, henna, lipstick, rag, red, scent, wednesday"
2,poem_12480,4,6,4,8,10,"curse, hay, heath, sun","devil, grain, haycock, pourd","heavy rain, increase, mercy, misery, peace, pity"
3,poem_06414,4,5,4,8,9,"collection, devoid, drop, limit","air, need, value, water","anthology, detour, invitation, reappear, rivulet"
4,poem_11839,6,3,2,8,9,"image, lens, photo, print, wall, worker","poet, room","contradiction, deed, identity"
5,poem_09632,4,6,4,8,10,"answer, eli, morning, synagogue","alley, house, lamp, study","believer, eternal life, poultry, prayer, reb, zion"
6,poem_10031,2,4,6,8,6,"current, reality","air, atmosphere, heel, season, silk, space","bacteria, billion, design, whine"
7,poem_01828,3,5,5,8,8,"gown, lamp, room","cross, grass, story, tree, window","christmas, christmas tree, flame, hurricane, wick"
8,poem_01791,7,4,1,8,11,"alm, grove, meek, painter, peninsula, sulphur, volcano",apostle,"hue, ore, tempestuous, wanderer"
9,poem_11907,4,6,4,8,10,"cloud, color, motionless, rochester","house, saidshe, skya, universalsimply","angel, evil, happiness, luck, milton, radiance"


## Emotion Extraction Evaluation

The system emotion categories are compared with the gold emotion categories from the same sampled poems.


In [5]:
emotions_df = pd.read_csv(EMOTIONS_PATH)
gold_emotion_sets = dict(zip(gold_sample['poem_id'], gold_sample['gold_emotion_categories'].map(split_label_list)))
predicted_emotion_sets = emotions_df[emotions_df['poem_id'].isin(gold_emotion_sets)].groupby('poem_id')['emotion_category'].apply(lambda values: {normalize_value(value) for value in values.dropna() if normalize_value(value)}).to_dict()
emotion_summary, emotion_details = set_metrics(gold_emotion_sets, predicted_emotion_sets)
display(emotion_summary)
emotion_details.head(10)


,true_positive,false_positive,false_negative,precision,recall,f1
0,221,5,0,0.977876,1.0,0.988814


,poem_id,true_positive,false_positive,false_negative,gold_count,predicted_count,matched_items,missed_items,extra_items
0,poem_11269,3,0,0,3,3,"hope, joy, mortality",,
1,poem_05452,7,0,0,7,7,"despair, envy, freedom, hope, love, melancholy, resilience",,
2,poem_12480,6,0,0,6,6,"freedom, joy, melancholy, peace, tenderness, transcendence",,
3,poem_06414,6,0,0,6,6,"anxiety, freedom, gratitude, melancholy, mortality, peace",,
4,poem_11839,2,0,0,2,2,"doubt, nostalgia",,
5,poem_09632,12,0,0,12,12,"despair, doubt, faith, fear, hope, love, mortality, nostalgia, regret, resilience, transcendence, wonder",,
6,poem_10031,8,0,0,8,8,"anxiety, fear, freedom, grief, hope, longing, love, nostalgia",,
7,poem_01828,7,0,0,7,7,"envy, fear, hope, melancholy, nostalgia, peace, transcendence",,
8,poem_01791,4,0,0,4,4,"desire, longing, peace, transcendence",,
9,poem_11907,2,0,0,2,2,"joy, transcendence",,


## Relation Quality Evaluation

The relation annotation file is kept because it contains sampled relation labels and reasons. The notebook calculates the relation quality rate directly from this file.


In [6]:
relation_sample = pd.read_csv(EVALUATION_DIR / 'relation_annotation_sample.csv')
relation_quality = pd.DataFrame([{
    'sample_size': len(relation_sample),
    'meaningful_count': int(relation_sample['manual_label'].sum()),
    'meaningful_rate': relation_sample['manual_label'].mean()
}])
display(relation_quality)
relation_sample[['source_symbol', 'target_emotion', 'title', 'author', 'token_distance', 'context_snippet', 'manual_label', 'objective_reason']].head(15)


,sample_size,meaningful_count,meaningful_rate
0,100,35,0.35


,source_symbol,target_emotion,title,author,token_distance,context_snippet,manual_label,objective_reason
0,tern,freedom,from Beachy Head,Charlotte Smith,8,"thy rough hollows echo to the voice Of the gray choughs, and ever restless daws, With clamor, not unlike the chiding hounds, While the l...",0,context does not show the symbol clearly
1,game,tenderness,A Bed above the Abyss: Amnesiac Notebook,Andrew Allport,2,"mind.” Only the kingdom of living names was missing there—bank, flagstone, sofa remained, but not the blur at the feeder, the undersea c...",0,context does not show the symbol clearly
2,hope,hope,On the Subject of Doctors,James Tate,0,"with the doctors, who are dying.",0,symbol and emotion are identical
3,white petal,freedom,Mummy of a Lady Named Jemutesonekh,Thomas James,5,NaN,0,missing context snippet
4,river,confusion,When Black Men Drown Their Daughters,Patricia Smith,3,"them unreadable, and the overwhelm easily disappears the men, their wiry heads glistening, then gulped. All that’s left is the fathers’ ...",0,context does not show the symbol clearly
5,plum,transcendence,Crepuscule with Paula,Charles North,9,NaN,0,missing context snippet
6,nest,fear,The Yellowhammer's Nest,John Clare,8,"us stoop And seek its nest—the brook we need not dread, 'Tis scarcely deep enough a bee to drown, So it sings harmless o'er its pebbly b...",1,within 10-token window with visible symbol evidence
7,salute,mortality,Morituri Salutamus: Poem for the Fiftieth Anniversary of the Class of 1825 in Bowdoin College,Henry Wadsworth Longfellow,2,Ascends the ladder leaning on the cloud! As ancient Priam at the Scæan gate Sat on the walls of Troy in regal state,0,context does not show the symbol clearly
8,balm,loneliness,Portrait of a Figure near Water,Jane Kenyon,6,NaN,0,missing context snippet
9,poet,love,Careless Perfection,Daniel Halpern,8,"admired"" Tao Yuanming, a poet of nature who wrote a single love poem, a poem thought by Chinese dilettantes to be the one ""blemish in a ...",1,within 10-token window with visible symbol evidence


## Distance Threshold Comparison

This table recalculates relation quality for smaller token-distance windows using the same annotated relation sample.


In [7]:
threshold_rows = []
for threshold in [3, 5, 10]:
    subset = relation_sample[relation_sample['token_distance'] <= threshold]
    threshold_rows.append({
        'threshold': threshold,
        'sample_size': len(subset),
        'meaningful_count': int(subset['manual_label'].sum()) if len(subset) else 0,
        'meaningful_rate': subset['manual_label'].mean() if len(subset) else 0
    })
pd.DataFrame(threshold_rows)


,threshold,sample_size,meaningful_count,meaningful_rate
0,3,42,9,0.214286
1,5,61,20,0.327869
2,10,100,35,0.350000


## Retrieval Evaluation

The retrieval annotation file is kept because it contains the query, retrieved poem, similarity score, and relevance label. The notebook calculates retrieval quality directly from this file.


In [8]:
retrieval_sample = pd.read_csv(EVALUATION_DIR / 'retrieval_evaluation_sample.csv')
retrieval_rows = []
for query, group in retrieval_sample.groupby('query'):
    top = group.sort_values('similarity', ascending=False).head(5)
    retrieval_rows.append({
        'query': query,
        'precision_in_top_five': (top['manual_relevance'] >= 1).mean(),
        'mean_relevance_in_top_five': top['manual_relevance'].mean(),
        'relevant_count_in_top_five': int((top['manual_relevance'] >= 1).sum())
    })
retrieval_summary = pd.DataFrame(retrieval_rows)
overall_row = pd.DataFrame([{
    'query': 'overall',
    'precision_in_top_five': retrieval_summary['precision_in_top_five'].mean(),
    'mean_relevance_in_top_five': retrieval_summary['mean_relevance_in_top_five'].mean(),
    'relevant_count_in_top_five': retrieval_summary['relevant_count_in_top_five'].mean()
}])
display(pd.concat([retrieval_summary, overall_row], ignore_index=True))
retrieval_sample[['query', 'title', 'author', 'similarity', 'manual_relevance', 'annotation_method']].head(20)


,query,precision_in_top_five,mean_relevance_in_top_five,relevant_count_in_top_five
0,light hope,1.0,1.00,5.0
1,moon grief,1.0,1.60,5.0
2,sea longing,1.0,2.00,5.0
3,winter memory,1.0,1.20,5.0
4,overall,1.0,1.45,5.0


,query,title,author,similarity,manual_relevance,annotation_method
0,moon grief,Reasons,Thomas James,0.591225,2,objective_proxy_query_overlap_and_similarity
1,moon grief,Half Omen Half Hope,Joanna Klink,0.573256,2,objective_proxy_query_overlap_and_similarity
2,moon grief,Written with a Pencil Found in Lorine Niedecker’s Front Yard,David Trinidad,0.555862,2,objective_proxy_query_overlap_and_similarity
3,moon grief,October,Jacob Polley,0.540797,1,objective_proxy_query_overlap_and_similarity
4,moon grief,The Fish,William Butler Yeats,0.540683,1,objective_proxy_query_overlap_and_similarity
5,sea longing,Virginity,Anna Swir,0.623222,2,objective_proxy_query_overlap_and_similarity
6,sea longing,She longed to be an island,Marjorie Agosín,0.609021,2,objective_proxy_query_overlap_and_similarity
7,sea longing,Le Séducteur,Laura Mullen,0.600173,2,objective_proxy_query_overlap_and_similarity
8,sea longing,Self-Dependence,Matthew Arnold,0.599909,2,objective_proxy_query_overlap_and_similarity
9,sea longing,Undertow,Dean Young,0.598744,2,objective_proxy_query_overlap_and_similarity


## App Search Checks

The app search check is calculated live from graph files. It is not saved as a separate CSV file.


In [9]:
nodes_df = pd.read_csv(GRAPH_NODES_PATH)
relations_df = pd.read_csv(RELATIONS_PATH)
queries = ['moon', 'sea', 'grief', 'love', 'winter', 'light', 'death', 'memory', 'bird', 'loneliness']
app_rows = []
for query in queries:
    lowered = query.lower()
    node_matches = nodes_df[nodes_df['label'].fillna('').str.lower().str.contains(lowered, regex=False)]
    relation_matches = relations_df[
        relations_df['source_symbol'].fillna('').str.lower().str.contains(lowered, regex=False)
        | relations_df['target_emotion'].fillna('').str.lower().str.contains(lowered, regex=False)
    ]
    app_rows.append({
        'query': query,
        'matching_nodes': len(node_matches),
        'matching_relations': len(relation_matches),
        'top_related_node': node_matches['label'].iloc[0] if len(node_matches) else '',
        'example_poem_found': bool(len(relation_matches))
    })
pd.DataFrame(app_rows)


,query,matching_nodes,matching_relations,top_related_node,example_poem_found
0,moon,79,542,moon,True
1,sea,164,1243,sea,True
2,grief,8,5328,grief,True
3,love,357,13193,love,True
4,winter,84,145,winter,True
5,light,202,1803,light,True
6,death,87,765,death,True
7,memory,39,266,memory,True
8,bird,87,577,bird,True
9,loneliness,5,5451,loneliness,True


## Summary Interpretation

The evaluation covers these parts of the project pipeline.

- Symbol extraction is evaluated with precision, recall, and F1 against the gold symbol sample.
- Emotion extraction is evaluated with precision, recall, and F1 against the gold emotion category sample.
- Relation extraction is evaluated by the percentage of sampled relations with clean context evidence.
- Retrieval is evaluated with precision and mean relevance in the top five results.

The most important limitation is that these labels are objective proxy annotations, not expert literary annotations. For a stronger final report, the same annotation files can be manually reviewed by a human annotator and the notebook will still calculate the same metrics.
